In [1]:
from image_encoding_helper import get_model, encode_images, reconstruct_images
from tqdm import tqdm

c:\Users\aravi\miniconda3\envs\idl-project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ae_model = get_model()

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


c:\Users\aravi\miniconda3\envs\idl-project\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\aravi\miniconda3\envs\idl-project\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips\vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Restored from model.ckpt with 0 missing and 0 unexpected keys
Restored from model.ckpt with 0 missing and 0 unexpected keys


In [3]:
encode_images(ae_model, "RawImages", "EncodedImages", 2)

100%|█████████████████████████████████████| 15000/15000 [27:29<00:00,  9.09it/s]


In [ ]:
# reconstruct_images(ae_model, "EncodedImages", "ReconstuctedImages")

In [7]:
import os
import torch
import zipfile
import pickle
from transformers import BertTokenizer, BertModel

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

# Function to tokenize text and get embeddings
def get_embeddings(texts):
    # Tokenize input text
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        # Forward pass through BERT model
        outputs = model(**inputs)
    # Extract embeddings (last layer's output)
    embeddings = outputs.last_hidden_state
    return embeddings

# Directory to save output embeddings
output_dir = "celeba-caption\\celeba-caption"

# Dictionary to store embeddings
embeddings_dict = {}

# Batch size
batch_size = 10

# List to store filenames and texts
filenames = []
texts = []

# Iterate through each text file
for filename in tqdm(os.listdir(output_dir)):
    if filename.endswith(".txt"):
        # Read text from file
        with open(os.path.join(output_dir, filename), "r", encoding="utf-8") as file:
            text = file.read()
        
        # Append filename and text to lists
        filenames.append(filename)
        texts.append(text)
        
        # If batch size is reached, process the batch
        if len(texts) == batch_size:
            # Tokenize text and get embeddings
            embeddings = get_embeddings(texts)
            
            # Remove batch dimension and rearrange tensor
            embeddings = embeddings.squeeze(0)
            
            # Save embeddings to dictionary
            for i, filename in enumerate(filenames):
                embeddings_dict[filename] = embeddings[i]
            
            # Clear lists
            filenames = []
            texts = []

# Process remaining files in the last batch
if texts:
    # Tokenize text and get embeddings
    embeddings = get_embeddings(texts)
    
    # Remove batch dimension and rearrange tensor
    embeddings = embeddings.squeeze(0)
    
    # Save embeddings to dictionary
    for i, filename in enumerate(filenames):
        embeddings_dict[filename] = embeddings[i]

# Save dictionary to a .pkl file
output_file_path = "embeddings_dict.pkl"
with open(output_file_path, "wb") as output_file:
    pickle.dump(embeddings_dict, output_file)

 23%|██▎       | 6829/30000 [20:11<1:41:55,  3.79it/s]